# MVO4 Swan 10D: REPLICATION_GUIDE.md implementation

This revised notebook uses the shared `mvo_algo.py` pipeline. Run from the `replication` directory. The uploaded notebook is preserved separately. The guide controls all dates, swans, and model choices. No live downloads are used.

In [1]:
import os, sys
from pathlib import Path
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "1"
HERE = Path.cwd()
if not (HERE / "mvo_algo.py").exists():
    raise RuntimeError("Run this notebook from the replication directory")
sys.path.insert(0, str(HERE))
import pandas as pd
import matplotlib.pyplot as plt
from mvo_algo import Replication, EVENTS, versions, write_outputs
model = Replication(cache=HERE / ".cache/dbi")
versions()

{'python': '3.12.14',
 'numpy': '2.2.3',
 'pandas': '2.3.3',
 'scikit-learn': '1.7.2',
 'scipy': '1.16.3',
 'PyPortfolioOpt': '1.6.0',
 'cvxpy': '1.9.2'}

## 1–3. Historical inputs, plain-average crash shape, and inclusive window

In [2]:
EVENT = 'Swan 10D'
K, Seeds, MinAppear, w1 = 7, 3, 3, 0.17
ev = EVENTS[EVENT]
print(ev)
print("Eligible universe:", len(model.training_beta(EVENT)))
print("t1, t2, dd:", model.curve_params(EVENT))
print("start, end, extended:", model.training_window(EVENT))

Event(name='Swan 10D', peak='2022-01-03', bottom='2022-10-12', train_swans=('Swan5', 'Swan6', 'Swan7', 'Swan8', 'Swan9'), optdata='revised_swan10d_sp100_optdata4.xlsx')
Eligible universe: 92
t1, t2, dd: (66, 188, 0.25467047442812923)
start, end, extended: (Timestamp('2020-11-15 00:00:00'), Timestamp('2021-12-31 00:00:00'), False)


## 4. Rank all 1,000 seeds, vote, retain the top 22 with at least MinAppear votes

In [3]:
ranking = model.dbi_ranking(EVENT, K)
print("Best seeds:", ranking[:Seeds])
selected, votes = model.preselect(EVENT, K, Seeds, MinAppear)
display(votes.to_frame())
display(selected)
assert len(selected) >= 5, "This combination has no_preselect status"

Best seeds: [(574, 0.9922642011385697), (729, 1.0105741889937727), (873, 1.0105741889937727)]


,votes
AMT,3
DUK,3
EXC,3
KO,3
LLY,3
MCD,3
MDLZ,3
MRK,3
NEE,3
PEP,3


,Swan5,Swan6,Swan7,Swan8,Swan9,avg
Ticker,,,,,,
AMT,0.583651,0.050751,0.583639,-0.336659,0.882155,0.352708
DUK,0.771714,-0.322793,0.233701,-0.356250,1.165454,0.298365
EXC,0.794675,0.263389,0.605246,-0.123438,1.254518,0.558878
KO,0.536675,0.325968,0.614677,-0.049526,1.161009,0.517761
LLY,0.920577,0.280671,0.780342,0.061466,0.490481,0.506707
MCD,0.321829,0.130072,0.483323,-0.202955,1.142392,0.374932
MDLZ,0.828315,0.344205,0.089729,0.386481,0.928976,0.515541
MRK,0.756485,0.674926,0.995745,-0.019569,0.582136,0.597945
NEE,0.503894,0.415367,0.192033,-0.026635,1.062981,0.429528


## 5–6. Inject at the front, then compute last-year drawdown and weekly covariance ×51

In [4]:
risk = model.risk_inputs(EVENT, selected)
print("Trading days:", risk["total_days"], "Cutoff:", risk["cutoff"])
pd.Series(risk["curve"], index=risk["raw_prices"].index).plot(title="Injected market drawdown")
plt.show()
display(pd.DataFrame({"fall": risk["fall"], "R_fwd_30_MA": risk["R_fwd"], "E_2": risk["E_2"]}))
display(risk["cov"])

Trading days: 284 Cutoff: 2020-12-31 00:00:00


,fall,R_fwd_30_MA,E_2
Ticker,,,
AMT,0.147320,0.113587,0.771017
DUK,0.098415,0.053651,0.545152
EXC,0.118469,0.062917,0.531085
KO,0.158084,0.113038,0.715052
LLY,0.183171,0.007401,0.040407
MCD,0.063454,0.059948,0.944745
MDLZ,0.139036,0.123849,0.890767
MRK,0.200532,0.210639,1.050402
NEE,0.171969,0.044532,0.258952


Ticker,AMT,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,VZ
Ticker,,,,,,,,,,,,,,
AMT,0.044439,0.013625,0.015481,0.010372,-0.001690,0.005476,0.010002,0.000932,0.019587,0.011667,0.007896,0.013927,0.011829,0.007162
DUK,0.013625,0.022749,0.015415,0.010627,0.007854,0.001360,0.011180,0.007452,0.010260,0.013890,0.010299,0.005475,0.011494,0.005690
EXC,0.015481,0.015415,0.030879,0.012654,0.008227,0.010295,0.014737,0.010394,0.019814,0.015374,0.014294,0.013664,0.013962,0.005520
KO,0.010372,0.010627,0.012654,0.030675,0.004630,0.013012,0.019033,0.010466,0.004072,0.020475,0.016539,0.016381,0.014476,0.008757
LLY,-0.001690,0.007854,0.008227,0.004630,0.098632,-0.007096,0.003695,0.018313,0.014712,-0.001155,-0.002667,0.000881,0.001479,0.005134
MCD,0.005476,0.001360,0.010295,0.013012,-0.007096,0.019938,0.011249,0.002856,0.003736,0.011018,0.009268,0.018955,0.009174,0.001450
MDLZ,0.010002,0.011180,0.014737,0.019033,0.003695,0.011249,0.024662,0.009324,0.004104,0.017613,0.015605,0.011168,0.012109,0.009383
MRK,0.000932,0.007452,0.010394,0.010466,0.018313,0.002856,0.009324,0.063804,0.002748,0.005322,0.004570,-0.009881,0.000107,0.014438
NEE,0.019587,0.010260,0.019814,0.004072,0.014712,0.003736,0.004104,0.002748,0.043584,0.009330,0.009817,0.014905,0.016160,-0.000653


## 7–8. Max Sharpe at rf=0, 3% floor, then buy at peak and evaluate at bottom

In [5]:
result = model.run_one(EVENT, K, Seeds, MinAppear, w1, detail=True)
print("Status:", result["status"], "PR:", result["PR"])
assert result["status"] == "ok", result.get("error")
display(result["weights"].to_frame("weight"))
pd.DataFrame({"portfolio": result["evaluation"]["portfolio_values"], "benchmark": result["evaluation"]["benchmark_values"]}).plot(title=EVENT)
plt.show()

Status: ok PR: 0.4460941446207144


,weight
AMT,0.094129
DUK,0.090975
MCD,0.174656
MDLZ,0.167937
MRK,0.171215
SBUX,0.126431
VZ,0.174656


## 9. Small sweep and optional exports

The command-line runner handles the complete 5,376-run grid. Below is a four-combination event sweep; failed optimization is retained as a status, with no alternate model.

In [6]:
small = model.run_sweep(dict(K=[7], Seeds=[3], MinAppear=[3], w1=[0.05, 0.11, 0.15, 0.17]), [EVENT])
display(small)
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output = HERE / "results" / EVENT.replace(" ", "_")
    write_outputs(small, output, [EVENT])
    model.export_checkpoints(output / "replication_checkpoints.xlsx", K=K, Seeds=Seeds, MinAppear=MinAppear, w1=w1)

Swan 10D, K=7: 4 runs completed


,event,K,Seeds,MinAppear,w1,status,PR,port_dd,market_dd,n_preselect,n_final,holdings,error,train_start,train_end
0,Swan 10D,7,3,3,0.05,opt_failed,NaN,NaN,NaN,14,0,{},infeasible: n_stocks * w1 < 1,2020-11-15,2021-12-31
1,Swan 10D,7,3,3,0.11,ok,0.446791,0.150638,0.272298,14,9,"{""AMT"": 0.11196156664766714, ""DUK"": 0.11196156...",None,2020-11-15,2021-12-31
2,Swan 10D,7,3,3,0.15,ok,0.439665,0.152578,0.272298,14,8,"{""AMT"": 0.09776, ""DUK"": 0.11116, ""MCD"": 0.15, ...",None,2020-11-15,2021-12-31
3,Swan 10D,7,3,3,0.17,ok,0.446094,0.150828,0.272298,14,7,"{""AMT"": 0.09412949226375161, ""DUK"": 0.09097540...",None,2020-11-15,2021-12-31


Limitations: original checkpoint workbook is missing; fresh Yahoo downloads and alternative experimental objectives are outside this replication workflow. Swan9 analyst inputs have the guide's point-in-time caveat. See `IMPLEMENTATION_REPORT.md` for validation results.